<a href="https://colab.research.google.com/github/dee431/-AI-Powered-Audio-Recommendation-Engine-Dynamic-Web-Host/blob/main/Personal_Finance_tracker%F0%9F%AA%99%F0%9F%92%B3%F0%9F%92%B4%F0%9F%92%B8%F0%9F%92%B2%F0%9F%92%B6%F0%9F%AA%99.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Cell 1: Setup and Imports**

In [1]:
# Cell 1: Install necessary libraries (if running locally, otherwise Colab has these pre-installed)
!pip install plotly ipywidgets scikit-learn pandas numpy

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML
from datetime import datetime, timedelta
from sklearn.ensemble import GradientBoostingRegressor, IsolationForest
from sklearn.model_selection import train_test_split

# Set a consistent, modern color palette (Blue/Purple/Green)
COLORS = {
    'income': '#10b981', 'expense': '#ef4444', 'primary': '#6366f1',
    'housing': '#3b82f6', 'food': '#f59e0b', 'transport': '#8b5cf6',
    'utilities': '#ec4899', 'other': '#6b7280', 'saved': '#10b981'
}

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 29.1 MB/s eta 0:00:00


# **Cell 2: Smart Synthetic Dataset Generation**

In [2]:
# Cell 2: Generate a highly realistic synthetic dataset
np.random.seed(42)

dates = pd.date_range(start='2026-01-01', end='2026-08-31', freq='D')
data = []

for date in dates:
    # Monthly patterns
    day = date.day

    # Income (Monthly Salary)
    if day == 1:
        data.append([date, 'MONTHLY SALARY', 'INCOME', 4800])
    if day == 15:
        data.append([date, 'FREELANCE PROJECT', 'INCOME', 300])

    # Expenses (Recurring + Variable)
    if day == 5: data.append([date, 'INTERNET SERVICE', 'UTILITIES', 75])
    if day == 10: data.append([date, 'RENT PAYMENT', 'HOUSING', 1350])
    if day == 12: data.append([date, 'TRAIN PASS', 'TRANSPORT', 90])
    if day == 20: data.append([date, 'NETFLIX', 'UTILITIES', 15.99])

    # Random Daily Expenses
    if np.random.rand() > 0.4: # 60% chance of spending
        cat = np.random.choice(['FOOD', 'TRANSPORT', 'OTHER', 'FOOD', 'FOOD'])
        amt = {
            'FOOD': np.random.uniform(5, 60),
            'TRANSPORT': np.random.uniform(10, 30),
            'OTHER': np.random.uniform(5, 150),
            'FOOD': np.random.uniform(5, 60)
        }[cat]
        data.append([date, 'GROCERY MART' if cat=='FOOD' else 'TAXI' if cat=='TRANSPORT' else 'MISC SHOPPING', cat, round(amt, 2)])

# Create DataFrame
df = pd.DataFrame(data, columns=['date', 'description', 'category', 'amount'])
df['type'] = df['description'].apply(lambda x: 'INCOME' if 'SALARY' in x or 'FREELANCE' in x else 'EXPENSE')

# Clean up
df['date'] = pd.to_datetime(df['date'])
df['month'] = df['date'].dt.strftime('%b')
df['year'] = df['date'].dt.year
df['day'] = df['date'].dt.day
df['weekday'] = df['date'].dt.weekday

print(f"Dataset generated with {len(df)} transactions.")
df.head()

Dataset generated with 183 transactions.


,date,description,category,amount,type,month,year,day,weekday
0,2026-01-01,MONTHLY SALARY,INCOME,4800.00,INCOME,Jan,2026,1,3
1,2026-01-02,MISC SHOPPING,OTHER,69.65,EXPENSE,Jan,2026,2,4
2,2026-01-03,GROCERY MART,FOOD,58.35,EXPENSE,Jan,2026,3,5
3,2026-01-04,TAXI,TRANSPORT,13.67,EXPENSE,Jan,2026,4,6
4,2026-01-05,INTERNET SERVICE,UTILITIES,75.00,EXPENSE,Jan,2026,5,0


# **Cell 3: Advanced ML Engineering (Forecasting and Anomaly Detection)**

In [3]:
# Cell 3: Feature Engineering & Machine Learning Model
# Split data into Income and Expenses
expenses = df[df['type'] == 'EXPENSE'].copy()
income = df[df['type'] == 'INCOME'].copy()

# Aggregate daily expenses for time-series model
daily_expenses = expenses.groupby('date')['amount'].sum().reset_index()
daily_expenses.set_index('date', inplace=True)
daily_expenses = daily_expenses.resample('D').sum().fillna(0)

# Create lag features for ML Model
daily_expenses['lag_1'] = daily_expenses['amount'].shift(1)
daily_expenses['lag_7'] = daily_expenses['amount'].shift(7)
daily_expenses['rolling_mean_3'] = daily_expenses['amount'].rolling(window=3).mean()
daily_expenses = daily_expenses.dropna()

# Train Gradient Boosting Regressor to predict next day's spending
X = daily_expenses[['lag_1', 'lag_7', 'rolling_mean_3']]
y = daily_expenses['amount']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3)
model.fit(X_train, y_train)

# Anomaly Detection using Isolation Forest (Flag unusual huge expenses)
anomaly_detector = IsolationForest(contamination=0.05, random_state=42)
expenses['anomaly'] = anomaly_detector.fit_predict(expenses[['amount']])
expenses['anomaly'] = expenses['anomaly'].apply(lambda x: 'Outlier' if x == -1 else 'Normal')

print(f"Training Complete. Model R2 Score: {model.score(X_test, y_test):.2f}")
print("Outlier Detection initialized.")

Training Complete. Model R2 Score: 0.96
Outlier Detection initialized.


# **Cell 4: Interactive Dashboard Layout (KPI Cards & Filters)**

In [4]:
# Cell 4: Dashboard Part 1 - KPI Cards and Interactive Filters
# Create a dropdown to filter by Month
months = ['All'] + sorted(df['month'].unique().tolist())
month_dropdown = widgets.Dropdown(options=months, value='All', description='Month:')

# Define a function to update dashboard based on month
def update_dashboard(selected_month):
    filtered_df = df.copy()
    if selected_month != 'All':
        filtered_df = filtered_df[filtered_df['month'] == selected_month]

    total_income = filtered_df[filtered_df['type']=='INCOME']['amount'].sum()
    total_expense = filtered_df[filtered_df['type']=='EXPENSE']['amount'].sum()
    total_saved = total_income - total_expense

    # HTML for KPI Cards (Styled to match the image)
    kpi_html = f"""
    <div style="display: flex; justify-content: space-between; margin-bottom: 20px;">
        <div style="background: linear-gradient(135deg, #f3f4f6, #ffffff); border-radius: 10px; padding: 15px; width: 23%; box-shadow: 2px 2px 5px rgba(0,0,0,0.1);">
            <h5 style="color: #6b7280; margin: 0;">TOTAL INCOME</h5>
            <h2 style="color: {COLORS['income']}; margin: 5px 0;">${total_income:,.0f}</h2>
        </div>
        <div style="background: linear-gradient(135deg, #f3f4f6, #ffffff); border-radius: 10px; padding: 15px; width: 23%; box-shadow: 2px 2px 5px rgba(0,0,0,0.1);">
            <h5 style="color: #6b7280; margin: 0;">TOTAL EXPENSES</h5>
            <h2 style="color: {COLORS['expense']}; margin: 5px 0;">${total_expense:,.0f}</h2>
        </div>
        <div style="background: linear-gradient(135deg, #f3f4f6, #ffffff); border-radius: 10px; padding: 15px; width: 23%; box-shadow: 2px 2px 5px rgba(0,0,0,0.1);">
            <h5 style="color: #6b7280; margin: 0;">AMOUNT SAVED</h5>
            <h2 style="color: {COLORS['primary']}; margin: 5px 0;">${total_saved:,.0f}</h2>
        </div>
        <div style="background: linear-gradient(135deg, #f3f4f6, #ffffff); border-radius: 10px; padding: 15px; width: 23%; box-shadow: 2px 2px 5px rgba(0,0,0,0.1);">
            <h5 style="color: #6b7280; margin: 0;">OVERVIEW</h5>
            <h5 style="color: #6366f1; margin: 5px 0;">{selected_month if selected_month != 'All' else 'YTD'}</h5>
        </div>
    </div>
    """
    display(HTML(kpi_html))
    return filtered_df

# Generate interactive output
interactive_output = widgets.interactive_output(update_dashboard, {'selected_month': month_dropdown})
display(month_dropdown, interactive_output)

Dropdown(description='Month:', options=('All', 'Apr', 'Aug', 'Feb', 'Jan', 'Jul', 'Jun', 'Mar', 'May'), value=…

Output()

# **Cell 5: Dashboard Part 2 - Creative Charts (Line & Donut)**

In [17]:
# Cell 5: Dashboard Part 2 - Interactive Charts (Line Chart and Donut Chart)
def create_charts(filtered_df):
    # Data Aggregation for Charts
    monthly_agg = df.groupby(['month', 'type'])['amount'].sum().reset_index()

    # 1. Line Chart: Income vs Expenses
    fig_line = go.Figure()
    for t in ['INCOME', 'EXPENSE']:
        sub = monthly_agg[monthly_agg['type'] == t]
        fig_line.add_trace(go.Scatter(
            x=sub['month'], y=sub['amount'], mode='lines+markers',
            name=t.capitalize(), line=dict(color=COLORS[t.lower()], width=3),
            marker=dict(size=8)
        ))

    # 2. Donut Chart: Spending by Category
    cat_agg = filtered_df[filtered_df['type']=='EXPENSE'].groupby('category')['amount'].sum().reset_index()
    fig_donut = go.Figure(data=[go.Pie(
        labels=cat_agg['category'], values=cat_agg['amount'], hole=.6,
        marker=dict(colors=[COLORS.get(c.lower(), COLORS['other']) for c in cat_agg['category']]),
        textinfo='label+percent', hovertemplate="%{label}: $%{value:,.0f}<br>%{percent}"
    )])

    # Combine into subplots with correct 'specs'
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("Income vs. Expenses", "Spending by Category"),
        specs=[[{"type": "xy"}, {"type": "domain"}]]
    )

    for trace in fig_line.data:
        fig.add_trace(trace, row=1, col=1)
    for trace in fig_donut.data:
        fig.add_trace(trace, row=1, col=2)

    fig.update_layout(height=450, title_text="Monthly Insights Dashboard", showlegend=True, paper_bgcolor='white', plot_bgcolor='white')
    return fig

# **Cell 6: Detailed Insights, Anomalies and Recent Transactions**

In [18]:
# Cell 6: Recent Transactions Table, Insights and Savings Goals
def display_full_dashboard(selected_month):
    # 1. Update KPIs and get data
    filtered_df = update_dashboard(selected_month)

    # 2. Display Charts
    fig_charts = create_charts(filtered_df)
    fig_charts.show()

    # 3. Monthly Insight Calculation
    recent = filtered_df.sort_values(by='date', ascending=False).head(8)
    recent_display = recent.copy()
    recent_display['amount'] = recent_display['amount'].apply(lambda x: f"${x:,.2f}")

    # Table using Plotly
    fig_table = go.Figure(data=[go.Table(
        header=dict(values=['Date', 'Description', 'Category', 'Amount'], fill_color='#f9fafb', align='left'),
        cells=dict(values=[recent_display['date'].dt.strftime('%b %d'), recent_display['description'], recent_display['category'], recent_display['amount']],
                   fill_color='white', align='left'))
    ])
    fig_table.update_layout(height=300, margin=dict(l=0, r=0, t=0, b=0))

    # Display footer (Insights and Table)
    display(HTML(f"""
    <div style="display: flex; gap: 20px; margin-top: 20px;">
        <div style="flex: 1;">
            <div style="background: #f3f4f6; padding: 15px; border-radius: 10px;">
                <h4>Savings & Goals</h4>
                <p>Emergency Fund: $4,400 / $10,000</p>
                <div style="background: #e5e7eb; border-radius: 5px; height: 10px; width: 100%;">
                    <div style="background: {COLORS['primary']}; width: 44%; height: 10px; border-radius: 5px;"></div>
                </div>
                <p style="margin-top: 10px; font-size: 13px; color: #6b7280;">Tip: Your grocery spending is 12% lower this month!</p>
            </div>
        </div>
        <div style="flex: 2;">
            {fig_table.to_html(full_html=False, include_plotlyjs='cdn')}
        </div>
    </div>
    """))

# Re-bind the interactive output to the full dashboard function
full_interactive = widgets.interactive_output(display_full_dashboard, {'selected_month': month_dropdown})
display(month_dropdown, full_interactive)

Dropdown(description='Month:', index=3, options=('All', 'Apr', 'Aug', 'Feb', 'Jan', 'Jul', 'Jun', 'Mar', 'May'…

Output()

# **Cell 7: AI Future Prediction**

In [19]:
# Cell 7: Future Forecast Module (The "Best AI" feature)
def predict_next_month():
    # Forecast the next 30 days of expenses
    X_pred = pd.DataFrame({
        'lag_1': [daily_expenses['amount'].iloc[-1]],
        'lag_7': [daily_expenses['amount'].iloc[-7]],
        'rolling_mean_3': [daily_expenses['amount'].iloc[-3:].mean()]
    })

    future_expenses = []
    for _ in range(30):
        pred = model.predict(X_pred)[0]
        future_expenses.append(pred)
        # Update lag features for next iteration
        X_pred['lag_1'] = pred
        X_pred['lag_7'] = X_pred['lag_1'] # Simplified lag
        X_pred['rolling_mean_3'] = (X_pred['lag_1'] + X_pred['rolling_mean_3']*2)/3

    total_forecast = sum(future_expenses)

    # Visualize forecast
    forecast_dates = pd.date_range(start=df['date'].max() + timedelta(days=1), periods=30)
    fig_forecast = go.Figure()
    fig_forecast.add_trace(go.Scatter(x=df['date'], y=daily_expenses['amount'], mode='lines', name='Historical'))
    fig_forecast.add_trace(go.Scatter(x=forecast_dates, y=future_expenses, mode='lines', name='Forecast', line=dict(dash='dot', color='#8b5cf6')))
    fig_forecast.update_layout(title="AI Prediction: Next 30 Days Spending Forecast")
    fig_forecast.show()

    display(HTML(f"<h3>Next Month Predicted Total Expenses: <span style='color:#ef4444'>${total_forecast:,.2f}</span></h3>"))

predict_next_month()

# **Cell 8 Dashboard**

In [20]:
# CELL 7 (FINAL CELL): The Unique Creative Dashboard
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, HTML

# ---------------------------------------------------------
# 1. Prepare the exact data from the image
# ---------------------------------------------------------
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug']
income_data = [4000, 4000, 4000, 4000, 4000, 4000, 4000, 4800]
expense_data = [3500, 4000, 3000, 4500, 3500, 4000, 3000, 3260]

# Donut Data
donut_labels = ['HOUSING', 'FOOD', 'TRANSPORT', 'UTILITIES', 'OTHER']
donut_values = [1350, 620, 390, 350, 550]
donut_colors = ['#3b82f6', '#f59e0b', '#8b5cf6', '#ec4899', '#6b7280']

# Table Data
transactions = [
    {'date': 'AUG 18', 'desc': 'MONTHLY SALARY', 'cat': 'INCOME', 'amt': '+$4,800', 'class': 'amount-pos'},
    {'date': 'AUG 16', 'desc': 'GROCERY MART', 'cat': 'FOOD', 'amt': '-$146', 'class': 'amount-neg'},
    {'date': 'AUG 14', 'desc': 'INTERNET SERVICE', 'cat': 'UTILITIES', 'amt': '-$75', 'class': 'amount-neg'},
    {'date': 'AUG 12', 'desc': 'TRAIN PASS', 'cat': 'TRANSPORT', 'amt': '-$90', 'class': 'amount-neg'}
]

# ---------------------------------------------------------
# 2. Create the Interactive Charts
# ---------------------------------------------------------
# Line Chart: Income vs Expenses
fig_line = go.Figure()
fig_line.add_trace(go.Scatter(x=months, y=income_data, mode='lines+markers',
                              name='INCOME', line=dict(color='#6366f1', width=3),
                              marker=dict(size=8, color='#6366f1')))
fig_line.add_trace(go.Scatter(x=months, y=expense_data, mode='lines+markers',
                              name='EXPENSES', line=dict(color='#8b5cf6', width=3, dash='dot'),
                              marker=dict(size=8, color='#8b5cf6')))
fig_line.update_layout(
    title={'text': "", 'x': 0.5},
    height=300, margin=dict(l=10, r=10, t=10, b=10),
    paper_bgcolor='white', plot_bgcolor='white',
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    yaxis=dict(showgrid=True, gridcolor='#e5e7eb', range=[0, 6000]),
    xaxis=dict(showgrid=False)
)

# Donut Chart: Spending by Category
fig_donut = go.Figure(data=[go.Pie(
    labels=donut_labels, values=donut_values, hole=.6,
    marker=dict(colors=donut_colors),
    textinfo='label+value+percent',
    textposition='outside',
    hovertemplate="%{label}: $%{value:,.0f}<br>%{percent}"
)])
fig_donut.update_layout(
    height=300, margin=dict(l=10, r=10, t=10, b=10),
    paper_bgcolor='white', plot_bgcolor='white',
    showlegend=False,
    annotations=[dict(text="$3,260<br>Total Spent", x=0.5, y=0.5, font_size=14, showarrow=False)]
)

# Convert charts to HTML to embed them in the CSS grid
html_line_chart = pio.to_html(fig_line, full_html=False, include_plotlyjs='cdn')
html_donut_chart = pio.to_html(fig_donut, full_html=False, include_plotlyjs=False)

# ---------------------------------------------------------
# 3. Build the HTML layout with exact CSS styling
# ---------------------------------------------------------
dashboard_html = f"""
<style>
  .dashboard {{
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    background-color: #ffffff; color: #333; padding: 20px; max-width: 1000px; margin: 0 auto;
  }}
  .header {{
    display: flex; justify-content: space-between; align-items: center;
    border-bottom: 1px solid #f3f4f6; padding-bottom: 10px; margin-bottom: 20px;
  }}
  .header-title {{ font-weight: 800; font-size: 18px; letter-spacing: 1px; }}
  .progress-steps {{ display: flex; gap: 15px; align-items: center; }}
  .step {{ display: flex; align-items: center; gap: 5px; font-size: 11px; color: #6b7280; font-weight: bold; }}
  .step-dot {{ width: 16px; height: 16px; border-radius: 50%; border: 2px solid #6366f1; background: #6366f1; color: white; display: flex; justify-content: center; align-items: center; font-size: 9px; }}
  .step-line {{ width: 20px; height: 1px; background: #6366f1; }}

  .kpi-row {{ display: flex; gap: 15px; margin-bottom: 20px; }}
  .kpi-card {{ background: white; border-radius: 8px; padding: 15px; flex: 1; border: 1px solid #f3f4f6; }}
  .kpi-label {{ font-size: 11px; color: #9ca3af; font-weight: bold; margin-bottom: 5px; }}
  .kpi-value {{ font-size: 22px; font-weight: 800; }}

  .main-row {{ display: flex; gap: 15px; }}
  .col {{ flex: 1; display: flex; flex-direction: column; gap: 15px; }}
  .card {{ background: white; border-radius: 8px; padding: 15px; border: 1px solid #f3f4f6; }}

  .chart-title {{ font-size: 13px; font-weight: 800; color: #374151; margin-bottom: 10px; }}

  /* Table Styles */
  .tbl {{ width: 100%; border-collapse: collapse; }}
  .tbl th, .tbl td {{ padding: 10px; text-align: left; border-bottom: 1px solid #f3f4f6; font-size: 13px; }}
  .tbl th {{ color: #9ca3af; font-weight: 600; }}
  .amount-pos {{ color: #10b981; font-weight: bold; }}
  .amount-neg {{ color: #374151; font-weight: bold; }}

  /* Insight Box */
  .insight-box {{ background: #f9fafb; border-radius: 8px; padding: 15px; }}
  .goal-bar {{ background: #e5e7eb; border-radius: 10px; height: 8px; width: 100%; margin: 10px 0; }}
  .goal-fill {{ background: #10b981; height: 8px; border-radius: 10px; width: 64%; }}

  /* Color Classes */
  .c-blue {{ color: #3b82f6; }} .c-purple {{ color: #8b5cf6; }} .c-green {{ color: #10b981; }} .c-red {{ color: #ef4444; }}
</style>

<div class="dashboard">
  <!-- Header with Progress Bar -->
  <div class="header">
    <div class="header-title">PERSONAL FINANCE TRACKER</div>
    <div class="progress-steps">
      <div class="step"><div class="step-dot">✓</div> ADD TRANSACTIONS</div>
      <div class="step-line"></div>
      <div class="step"><div class="step-dot">✓</div> ORGANIZE DATA</div>
      <div class="step-line"></div>
      <div class="step"><div class="step-dot">✓</div> ANALYZE SPENDING</div>
      <div class="step-line"></div>
      <div class="step"><div class="step-dot">✓</div> VIEW INSIGHTS</div>
    </div>
  </div>

  <!-- KPI Cards -->
  <div class="kpi-row">
    <div class="kpi-card">
      <div class="kpi-label">TOTAL INCOME</div>
      <div class="kpi-value c-blue">$4,800</div>
    </div>
    <div class="kpi-card">
      <div class="kpi-label">TOTAL EXPENSES</div>
      <div class="kpi-value c-purple">$3,260</div>
    </div>
    <div class="kpi-card">
      <div class="kpi-label">AMOUNT SAVED</div>
      <div class="kpi-value c-green">$1,540</div>
    </div>
    <div class="kpi-card">
      <div class="kpi-label">AUGUST OVERVIEW</div>
      <div class="kpi-value c-blue">AUG</div>
    </div>
  </div>

  <!-- Main Charts Row -->
  <div class="main-row">
    <!-- Left Column -->
    <div class="col">
      <div class="card">
        <div class="chart-title">INCOME VS. EXPENSES</div>
        {html_line_chart}
      </div>
      <div class="card">
        <div class="chart-title">RECENT TRANSACTIONS</div>
        <table class="tbl">
          <tr><th>DATE</th><th>DESCRIPTION</th><th>CATEGORY</th><th>AMOUNT</th></tr>
          {''.join([f"<tr><td>{t['date']}</td><td>{t['desc']}</td><td>{t['cat']}</td><td class='{t['class']}'>{t['amt']}</td></tr>" for t in transactions])}
        </table>
      </div>
    </div>

    <!-- Right Column -->
    <div class="col">
      <div class="card">
        <div class="chart-title">SPENDING BY CATEGORY</div>
        {html_donut_chart}
      </div>
      <div class="card insight-box">
        <div class="chart-title" style="color: #374151;">MONTHLY INSIGHT</div>
        <p style="font-size: 13px; color: #6b7280;">Spending is <span style="color:red; font-weight:bold;">14% higher</span> than last month.<br>Reducing dining expenses by $80 would increase this month's savings to <span style="color:green; font-weight:bold;">$1,620</span>.</p>

        <div class="chart-title" style="color: #374151; margin-top: 20px;">SAVINGS GOAL</div>
        <p style="font-size: 12px; color: #6b7280; margin-bottom: 5px;">Emergency Fund</p>
        <p style="font-size: 14px; font-weight: bold; margin: 0;">$6,400 of $10,000</p>
        <div class="goal-bar"><div class="goal-fill"></div></div>
        <div style="display: flex; justify-content: space-between; font-size: 11px; color: #9ca3af;">
          <span>64% COMPLETE</span>
          <span>$3,600 REMAINING</span>
        </div>
      </div>
    </div>
  </div>
</div>
"""

# ---------------------------------------------------------
# 4. Render the complete dashboard
# ---------------------------------------------------------
display(HTML(dashboard_html))

DATE,DESCRIPTION,CATEGORY,AMOUNT
AUG 18,MONTHLY SALARY,INCOME,"+$4,800"
AUG 16,GROCERY MART,FOOD,-$146
AUG 14,INTERNET SERVICE,UTILITIES,-$75
AUG 12,TRAIN PASS,TRANSPORT,-$90
